In [14]:
import pandas as pd
import numpy as np
import re
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from metric import precision_at_recall

# подход к решению

- осмотреть датасет, выявить пропуски, дубликаты
- создать временные окна cookie
- построить признаки
- подобрать параметры для lightgbm
- сделать валидацию по времени
- сделать walk-forward по всем окнам, чтобы получить n_rounds_final
- применить бэггинг по 5 сидам на полном трейне
- провести честную проверку без утечек

In [15]:
random_state = 42
np.random.seed(42)

In [16]:
events = pd.read_csv('events.csv.gz', parse_dates = ['event_ts'])
train = pd.read_csv('train.csv', parse_dates = ['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('test.csv', parse_dates = ['cookie_created_at', 'window_start_ts', 'window_end_ts'])

# train.head()
# test.head()
# events.head()
# events.info()
print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


## осмотр датасета

In [17]:
print(f'train shape: {train.shape}, test shape: {test.shape}, events shape: {events.shape}')
print(f'доля ботов в train: {train.target.mean():.4f}', '\n')
print(f'train window: {train.window_start_ts.min()} — {train.window_start_ts.max()}')
print(f'test window: {test.window_start_ts.min()} — {test.window_start_ts.max()}')
print(f'диапазон event_ts: {events.event_ts.min()} — {events.event_ts.max()}', '\n')
print(events.eid.value_counts().sort_index(), '\n')
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print(events.user_agent.str.slice(0, 20).value_counts(), '\n')
print(f'пустых user_agent: {(events.user_agent.fillna('').str.strip() == '').sum()}', '\n')
print(f'дубликатов: {events.duplicated().sum()}', '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

train shape: (11091, 5), test shape: (4909, 4), events shape: (328905, 14)
доля ботов в train: 0.0811 

train window: 2026-04-06 00:00:00 — 2026-04-19 00:00:00
test window: 2026-04-20 00:00:00 — 2026-04-26 00:00:00
диапазон event_ts: 2026-04-06 00:02:57 — 2026-04-26 23:59:50 

eid
100    100402
200    120817
210     36517
220     17403
300     11316
301      6125
303      3081
400     19049
500      6267
900      7928
Name: count, dtype: int64 

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name:

## временные окна cookie

In [18]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on = 'cookie_id', how = 'inner')
    ev = ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]
    return ev

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

198436 89690


## построение признаков

- количественные колонки n_* : у ботов либо слишком низкие значения, либо аномально высокие
- доли типов событий ratio_* : построение поведенческого профиля (у ботов часто только просмотры)
- временные признаки : у ботов низкая вариативность интервалов и работают круглосуточно
- поисковая страница : боты часто листают 10+ страниц, а человек по 2-3
- курсор : у ботов часто либо нет координат движений курсора, либо ставят (0, 0)
- user-agent : наличие headless в user-agent - прямое указание на бота
- метаданные : недавно созданные cookie и сразу активность - признаки бота

In [19]:
# признаки

headless_re = re.compile(r'headless|phantom|selenium|puppeteer|playwright|scrapy|'
                         r'python|curl|wget|go-http|okhttp|java|httpclient|'
                         r'node-fetch|axios|libwww|bot|spider|crawler', re.I)
browser_re = re.compile(r'chrome|firefox|safari|edg|opera|yabrowser|trident|msie', re.I)
mobile_os_re = re.compile(r'android|iphone|ipad|ios|mobile', re.I)

event_names = ['item_view', 'search_results_view', 'photo_swipe',
               'seller_page_view', 'favorite_add', 'contact_phone_show',
               'contact_chat_open', 'contact_message_sent', 'login']

# captcha не добавлена в event_names, так как она появляется только вне временного окна

In [20]:
def build_features(ev, meta):
    g = ev.groupby('cookie_id')
    f = pd.DataFrame(index = g.size().index)
    f['n_events'] = g.size()
    f['n_item'] = g.item_id.nunique()
    f['n_category'] = g.item_category.nunique()
    f['n_location'] = g.item_location.nunique()
    f['n_seller_type'] = g.seller_type.nunique()
    f['n_eid'] = g.eid.nunique()
    f['n_event_name'] = g.event_name.nunique()
    f['n_ua'] = g.user_agent.nunique()
    f['n_platform'] = g.platform.nunique()
    f['n_query'] = g.search_query.nunique()
    
#   доли типов событий
    for name in event_names:
        f[f'ratio_{name}'] = g.apply(lambda x, n = name: (x.event_name == n).mean()).fillna(0)
    f['items_per_event'] = f['n_item'] / f['n_events'].clip(lower = 1)
    f['category_per_item'] = f['n_category'] / f['n_item'].clip(lower = 1)
    f['seller_per_item'] = f['n_seller_type'] / f['n_item'].clip(lower = 1)
    
#   временные признаки
    ev_sorted = ev.sort_values(['cookie_id', 'event_ts'])
    ev_sorted['dt'] = (ev_sorted.groupby('cookie_id')['event_ts'].diff().dt.total_seconds())

    def time_feats(x):
        dt = x['dt'].dropna().values
        if len(dt) == 0:
            return pd.Series({
                'dt_mean': 0.0, 'dt_std': 0.0, 'dt_median': 0.0,
                'dt_cv': 0.0, 'dt_min': 0.0, 'dt_max': 0.0,
                'even_ratio': 0.0, 'span_sec': 0.0,
                'events_per_min': 0.0, 'hour_entropy': 0.0,
                'n_unique_hours': 0,
            })
        span = (x['event_ts'].max() - x['event_ts'].min()).total_seconds()
        hours = x['event_ts'].dt.hour.value_counts(normalize = True)
        return pd.Series({
            'dt_mean': dt.mean(),
            'dt_std': dt.std(),
            'dt_median': float(np.median(dt)),
            'dt_cv': dt.std() / (dt.mean() + 1e-9),
            'dt_min': dt.min(),
            'dt_max': dt.max(),
            
#            доля интервалов, близких к медиане (признак механической регулярности — робот кликает ровно каждые n секунд)
            'even_ratio': float(np.mean(np.abs(dt - np.median(dt)) < 0.5)),
            
#            интенсивность
            'span_sec': span,
            'events_per_min': len(dt) / (span / 60 + 1e-9),
            
#            энтропия распределения по часам суток
            'hour_entropy': float(-(hours * np.log(hours + 1e-12)).sum()),
            
#            количество активных часов в сутка
            'n_unique_hours': int(x['event_ts'].dt.hour.nunique()),
        })
    
    t_feats = ev_sorted.groupby('cookie_id').apply(time_feats)
    f = f.join(t_feats)
    
#   поисковая страница
    sp = ev.dropna(subset=['search_page']).groupby('cookie_id')['search_page']
    f['search_page_mean'] = sp.mean()
    f['search_page_max'] = sp.max()
    f['search_page_std'] = sp.std()
    
#   курсор
    has_ptr = ev['pointer_x'].notna() & ev['pointer_y'].notna()
    f['ptr_ratio'] = has_ptr.groupby(ev['cookie_id']).mean()
    f['ptr_x_std'] = g['pointer_x'].std()
    f['ptr_y_std'] = g['pointer_y'].std()
    f['ptr_x_mean'] = g['pointer_x'].mean()
    f['ptr_zero_ratio'] = (((ev['pointer_x'] == 0) | (ev['pointer_y'] == 0)).groupby(ev['cookie_id']).mean())
    
#   user-agent
    ua_stats = g['user_agent'].agg(
        ua_headless_ratio = lambda s: s.fillna('').str.contains(headless_re).mean(),
        ua_browser_ratio = lambda s: s.fillna('').str.contains(browser_re).mean(),
        ua_mobile_ratio = lambda s: s.fillna('').str.contains(mobile_os_re).mean())
    f = f.join(ua_stats)
    
#   платформа (нормализированная)
    f['platform_top_ratio'] = g['platform'].apply(lambda s: s.fillna('NA').str.lower().value_counts(normalize = True).iloc[0])
    
#   метаданные
    m = meta.set_index('cookie_id').copy()
    f = f.join(m[['cookie_created_at', 'window_start_ts', 'window_end_ts']])
    f['cookie_age_days'] = ((f['window_start_ts'] - f['cookie_created_at']).dt.total_seconds() / 86400)
    f = f.drop(columns = ['cookie_created_at', 'window_start_ts', 'window_end_ts'])

#   постобработка
    f = f.replace([np.inf, -np.inf], 0).fillna(0)
    f = (meta[['cookie_id']].merge(f.reset_index(), on = 'cookie_id', how = 'left').fillna(0))
    
    return f

Xtr = build_features(ev_tr, train)
Xte = build_features(ev_te, test)
ytr = train.target.values

cookies_tr = Xtr['cookie_id'].values
cookies_te = Xte['cookie_id'].values
Xtr = Xtr.drop(columns = ['cookie_id'])
Xte = Xte.drop(columns = ['cookie_id'])

# остаются только общие колонки 
common_cols = [c for c in Xtr.columns if c in Xte.columns]
Xtr = Xtr[common_cols]
Xte = Xte[common_cols]
print('features:', len(common_cols))

C:\Users\olesy\AppData\Local\Temp\ipykernel_13360\2321263634.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  f[f'ratio_{name}'] = g.apply(lambda x, n = name: (x.event_name == n).mean()).fillna(0)
C:\Users\olesy\AppData\Local\Temp\ipykernel_13360\2321263634.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  f[f'ratio_{name}'] = g.apply(lambda x, n = name: (x.event_name == n).mean()).fillna(0)
C:\Users\olesy\App

features: 46


In [21]:
Xtr.shape, Xte.shape

((11091, 46), (4909, 46))

## параметры для обучения модели

In [22]:
lgb_params = dict(
    objective = 'binary',
    metric = 'auc',
    learning_rate = 0.05,
    num_leaves = 31,
    max_depth = -1,
    min_data_in_leaf = 20,
    feature_fraction = 0.8,
    bagging_fraction = 0.8,
    bagging_freq = 1,
    lambda_l1 = 0.0,
    lambda_l2 = 1.0,
    is_unbalance = True,
    verbose = -1,
    n_jobs = -1,
    seed = random_state,
)

n_rounds = 3000
early_stopping = 200

## валидация

        вадационная выборка по времени (последние 3 дня train), так как test по датам идет после train, чтобы модель не подглядывала в будущее

In [23]:
is_valid = train.window_start_ts.ge('2026-04-17').values

dtrain = lgb.Dataset(Xtr.loc[~is_valid], label = ytr[~is_valid])
dvalid = lgb.Dataset(Xtr.loc[is_valid], label = ytr[is_valid])

booster = lgb.train(lgb_params, dtrain, num_boost_round = n_rounds, 
                    valid_sets = [dvalid], valid_names = ['valid'], 
                    callbacks = [lgb.early_stopping(early_stopping, verbose = False), 
                                 lgb.log_evaluation(period = 200)])

print('best_iteration:', booster.best_iteration, '\n')

p_va = booster.predict(Xtr.loc[is_valid], num_iteration = booster.best_iteration)

print('P@R0.7 на валидации:', round(precision_at_recall(ytr[is_valid], p_va), 4))
print('ROC-AUC на валидации:', round(roc_auc_score(ytr[is_valid], p_va), 4))
print('доля ботов (уровень константы):', round(ytr[is_valid].mean(), 4))

[200]	valid's auc: 0.926431
best_iteration: 196 

P@R0.7 на валидации: 0.7832
ROC-AUC на валидации: 0.927
доля ботов (уровень константы): 0.082


In [24]:
uniq_days = np.sort(train.window_start_ts.unique())
print('даты окон:', [str(pd.Timestamp(d).date()) for d in uniq_days])

даты окон: ['2026-04-06', '2026-04-07', '2026-04-08', '2026-04-09', '2026-04-10', '2026-04-11', '2026-04-12', '2026-04-13', '2026-04-14', '2026-04-15', '2026-04-16', '2026-04-17', '2026-04-18', '2026-04-19']


## walk-forward по всем окнам

In [25]:
oof_scores = []
oof_aucs = []
best_iters = []
oof_details = []

# обучаем на всех окнах до дня i, валидация на дне i
for i in range(2, len(uniq_days)):
#   маски для трейна и валадационной выборки
    tr_mask = (train.window_start_ts < uniq_days[i]).values
    va_mask = (train.window_start_ts == uniq_days[i]).values
    if va_mask.sum() < 50:
        continue

#   создаем датасеты по маскам
    dtr = lgb.Dataset(Xtr[tr_mask], label = ytr[tr_mask])
    dva = lgb.Dataset(Xtr[va_mask], label = ytr[va_mask])

#   обучаем модель на dtr, валидируем на списке dva
    b = lgb.train(lgb_params, dtr, num_boost_round = n_rounds, valid_sets = [dva],
                  callbacks = [lgb.early_stopping(early_stopping, verbose = False)])

#   считаем вероятности на валидационной части, остановливаемся на количестве деревьев, 
#   которое показало лучший результат на метрике, чтобы не переобучиться
    p = b.predict(Xtr[va_mask], num_iteration = b.best_iteration)

#   считаем метрики на трейне
    s_pr = precision_at_recall(ytr[va_mask], p)
    s_auc = roc_auc_score(ytr[va_mask], p)

#   сохраняем результаты
    oof_scores.append(s_pr)
    oof_aucs.append(s_auc)
    best_iters.append(b.best_iteration)
    oof_details.append((pd.Timestamp(uniq_days[i]).date(), s_pr, s_auc, 
                        b.best_iteration, int(va_mask.sum()), float(ytr[va_mask].mean())))

    print(f'окно {pd.Timestamp(uniq_days[i]).date()}: '
          f'P@R0.7 = {s_pr:.4f}  AUC = {s_auc:.4f}  '
          f'iters = {b.best_iteration}  '
          f'(cookie = {va_mask.sum()}, доля ботов = {ytr[va_mask].mean():.3f})', '\n')

print('mean walk-forward P@R0.7:', round(np.mean(oof_scores), 4))
print('mean walk-forward ROC-AUC:', round(np.mean(oof_aucs), 4))
print('median best_iteration:', int(np.median(best_iters)) if best_iters else None)

# немного увеличиваем количество итераций, так как на полном трейне больше данных
n_rounds_final = int(np.median(best_iters) * 1.2) if best_iters else 800
print('final num_boost_round:', n_rounds_final)

окно 2026-04-08: P@R0.7 = 0.5000  AUC = 0.9269  iters = 65  (cookie = 834, доля ботов = 0.084) 

окно 2026-04-09: P@R0.7 = 0.6528  AUC = 0.9457  iters = 115  (cookie = 902, доля ботов = 0.073) 

окно 2026-04-10: P@R0.7 = 0.8169  AUC = 0.9371  iters = 153  (cookie = 912, доля ботов = 0.090) 

окно 2026-04-11: P@R0.7 = 0.5773  AUC = 0.9148  iters = 58  (cookie = 896, доля ботов = 0.088) 

окно 2026-04-12: P@R0.7 = 0.4234  AUC = 0.9195  iters = 7  (cookie = 817, доля ботов = 0.082) 

окно 2026-04-13: P@R0.7 = 0.5132  AUC = 0.9341  iters = 56  (cookie = 843, доля ботов = 0.063) 

окно 2026-04-14: P@R0.7 = 0.7313  AUC = 0.9345  iters = 132  (cookie = 826, доля ботов = 0.085) 

окно 2026-04-15: P@R0.7 = 0.7419  AUC = 0.9335  iters = 325  (cookie = 801, доля ботов = 0.081) 

окно 2026-04-16: P@R0.7 = 0.6286  AUC = 0.9485  iters = 159  (cookie = 740, доля ботов = 0.081) 

окно 2026-04-17: P@R0.7 = 0.8393  AUC = 0.9161  iters = 176  (cookie = 713, доля ботов = 0.093) 

окно 2026-04-18: P@R0.7 =

    разброс P@R0.7 от 0.4234 до 0.8393 говорит о том, что поведение ботов нестабильно

## бэггинг и честная оценка

In [26]:
n_seeds = 5

dtrain_full = lgb.Dataset(Xtr, label = ytr)
preds_te = np.zeros(len(Xte))

dtrain_holdout = lgb.Dataset(Xtr.loc[~is_valid], label = ytr[~is_valid])
p_va_bag_honest = np.zeros(is_valid.sum())

# модели для test (обучаем на полном трейне)
for seed in range(n_seeds):
    params = {**lgb_params, 'seed': seed, 'bagging_seed': seed, 'feature_fraction_seed': seed}
    b = lgb.train(params, dtrain_full, num_boost_round = n_rounds_final)
    preds_te += b.predict(Xte) / n_seeds

# модели для честной оценки (обучаем без валидационного окна)
for seed in range(n_seeds):
    params = {**lgb_params, 'seed': seed, 'bagging_seed': seed, 'feature_fraction_seed': seed}
    b = lgb.train(params, dtrain_holdout, num_boost_round = n_rounds_final)
    p_va_bag_honest += b.predict(Xtr.loc[is_valid]) / n_seeds

print('честная оценка P@R0.7:', round(precision_at_recall(ytr[is_valid], p_va_bag_honest), 4))

честная оценка P@R0.7: 0.7943


## сабмит

In [27]:
sub = pd.DataFrame({
    'cookie_id': cookies_te,
    'score': preds_te,
})

assert len(preds_te) == len(cookies_te) == len(test) and sub['score'].between(0, 1).all()
assert sub['cookie_id'].is_unique

sub.to_csv('submission.csv', index = False)
print(sub.head())

             cookie_id     score
0  ck_315fb710a0e371e7  0.002948
1  ck_a76ee3b3e3e522fd  0.466623
2  ck_94c9a4d382689e82  0.139099
3  ck_8eaf9509ad9462a0  0.001158
4  ck_9a88a5a989cb5bc6  0.019624
